# 手动数据打包备份与清理工具

该 Notebook 用于**手动指定日期**将 Parquet 数据源文件打包为 `tar.gz` 压缩文件，并在确认无误后清理原始文件以释放硬盘空间。

**建议的执行流程**：
1. 执行 `analyze_data()` 查看当天的收据收集概况，确认数据正常。
2. 执行 `execute_backup()` 进行 `tar.gz` 压缩打包。
3. (**危险操作**) 执行 `cleanup_raw_data()` 删除已经备份的原始散落文件夹。

In [ ]:
import os
import shutil
import tarfile
import pandas as pd
from datetime import datetime, timedelta, timezone
from pathlib import Path

# =============== 基本配置 ===============

# 需要备份的目标日期 (格式 YYYY-MM-DD)，例如 '2026-02-25'
# 留空字符串 '' 则默认备份【昨天】的数据
TARGET_DATE = '2026-02-25'

# 路径配置
data_root = Path(os.path.abspath('../data'))
raw_dir = data_root / 'raw'
backup_dir = data_root / 'backup'

if not TARGET_DATE:
    TARGET_DATE = (datetime.now(timezone.utc) - timedelta(days=1)).strftime('%Y-%m-%d')
    print(f"未指定日期，自动选择了昨天: {TARGET_DATE}")

# ======================================

## 1. 备份前数据质量分析

在打包前，先扫描该日期的所有子文件夹，快速统计各个币种的 parquet 文件数量、是否损坏以及时间跨度。

In [ ]:
def analyze_data():
    global TARGET_DATE
    print(f"🔍 开始分析 {TARGET_DATE} 的数据质量...\n")
    
    if not raw_dir.exists():
        print(f"❌ 原始数据目录不存在: {raw_dir}")
        return

    results = []

    for data_type in ['trades', 'orderbooks']:
        type_dir = raw_dir / data_type
        if not type_dir.exists(): continue

        # 扫描所有市场类型
        for market_type_dir in type_dir.glob('market_type=*'):
            market_type = market_type_dir.name.split('=')[1]
            
            # 扫描交易所
            for exchange_dir in market_type_dir.glob('exchange=*'):
                exchange = exchange_dir.name.split('=')[1]
                
                # 扫描币种
                for symbol_dir in exchange_dir.glob('symbol=*'):
                    symbol = symbol_dir.name.split('=')[1]
                    
                    # 定位到具体的日期文件夹
                    date_dir = symbol_dir / f"date={TARGET_DATE}"
                    if not date_dir.exists():
                        continue
                        
                    valid_count = 0
                    empty_count = 0
                    corrupted_count = 0
                    min_ts = float('inf')
                    max_ts = 0
                    
                    # 遍历该目录下的所有 parquet (包括可能被标志为 corrupted 的文件)
                    for file_path in date_dir.glob('*'):
                        if 'corrupted' in file_path.name:
                            corrupted_count += 1
                            continue
                            
                        if file_path.suffix == '.parquet':
                            try:
                                df = pd.read_parquet(file_path)
                                if df.empty:
                                    empty_count += 1
                                else:
                                    valid_count += 1
                                    # 尝试获取时间跨度
                                    ts_col = 'local_ts' if 'local_ts' in df.columns else 'exchange_ts'
                                    if ts_col in df.columns:
                                        file_min = df[ts_col].min()
                                        file_max = df[ts_col].max()
                                        if file_min < min_ts: min_ts = file_min
                                        if file_max > max_ts: max_ts = file_max
                            except Exception:
                                corrupted_count += 1
                    
                    # 格式化时间跨度
                    time_span = "N/A"
                    if valid_count > 0 and min_ts != float('inf'):
                        # 假设时间戳为毫秒
                        start_time = datetime.fromtimestamp(min_ts / 1000, timezone.utc).strftime('%H:%M:%S')
                        end_time = datetime.fromtimestamp(max_ts / 1000, timezone.utc).strftime('%H:%M:%S')
                        time_span = f"{start_time} - {end_time}"
                        
                    results.append({
                        'Data': data_type.capitalize(),
                        'Type': market_type,
                        'Exchange': exchange,
                        'Symbol': symbol,
                        'Normal': valid_count,
                        'Empty': empty_count,
                        'Corrupted': corrupted_count,
                        'Time Span': time_span
                    })

    if results:
        df_results = pd.DataFrame(results)
        display(df_results)
        print(f"\n✅ 共找到 {len(results)} 个币种在 {TARGET_DATE} 的数据收集记录。")
    else:
        print(f"⚠️ 未能在 {TARGET_DATE} 找到任何收集数据。")

# 运行分析函数
analyze_data()

## 2. 执行数据打包压缩

将所有属于 `TARGET_DATE` 的零散文件夹收集起来，打包成 `tar.gz` 存放在 `data/backup` 中。

In [ ]:
def execute_backup():
    global TARGET_DATE
    
    backup_dir.mkdir(parents=True, exist_ok=True)
    print(f"🚀 开始打包 {TARGET_DATE} 的数据文件...")
    
    target_paths = []
    if not raw_dir.exists(): return
        
    for data_type in ['trades', 'orderbooks']:
        type_dir = raw_dir / data_type
        if not type_dir.exists(): continue
            
        for path in type_dir.rglob(f"date={TARGET_DATE}"):
            if path.is_dir():
                target_paths.append(path)
                
    if not target_paths:
        print(f"⚠️ 没找到需要打包的数据。")
        return
        
    # 创建压缩包
    timestamp = datetime.now(timezone.utc).strftime('%H%M%S')
    tar_name = f"crypto_data_{TARGET_DATE}_{timestamp}.tar.gz"
    tar_path = backup_dir / tar_name
    
    print(f"📦 正在写入: {tar_path}...")
    with tarfile.open(tar_path, "w:gz") as tar:
        for p in target_paths:
            arcname = p.relative_to(raw_dir)
            tar.add(p, arcname=arcname)
            
    print(f"🎉 打包完成！大小: {tar_path.stat().st_size / (1024*1024):.2f} MB")
    return tar_path

# 运行打包
# generated_tar = execute_backup()

## 3. 清理已打包的遗留源文件 (请谨慎操作)

确认备份包 (`.tar.gz`) 存在且大小正常后，您可以调用以下函数将散落在 `data/raw/` 里的当天原始文件彻底删除！这会有效释放服务器高达 60GB 的硬盘空间。

In [ ]:
def cleanup_raw_data():
    global TARGET_DATE
    
    print(f"⚠️ 警告：准备删除 {raw_dir} 下所有属于 {TARGET_DATE} 目录的数据！")
    
    # 简单的二次确认（防呆）
    user_input = input(f"请输入 'yes' 确认删除 {TARGET_DATE} 的数据文件：")
    if user_input.strip().lower() != 'yes':
        print("❌ 已取消删除操作。")
        return
        
    deleted_count = 0
    freed_space = 0
    
    for data_type in ['trades', 'orderbooks']:
        type_dir = raw_dir / data_type
        if not type_dir.exists(): continue
            
        # 扫描并删除 date={TARGET_DATE} 的文件夹
        for path in type_dir.rglob(f"date={TARGET_DATE}"):
            if path.is_dir():
                try:
                    # 递归计算该文件夹大小
                    folder_size = sum(f.stat().st_size for f in path.rglob('*') if f.is_file())
                    shutil.rmtree(path)
                    deleted_count += 1
                    freed_space += folder_size
                    print(f"  [-] 已删除: {path.relative_to(raw_dir)}")
                except Exception as e:
                    print(f"  [!] 无法删除 {path}: {e}")
                    
    if deleted_count > 0:
        print(f"\n🗑️ 清理完毕！共删除了 {deleted_count} 个文件夹，为您腾出了 {freed_space / (1024*1024):.2f} MB 的硬盘空间。")
    else:
        print("ℹ️ 没有找到需要删除的文件。")

# 如果确认您的包裹打好了，取消下面的注释来执行清理
# cleanup_raw_data()

---
## 附加功能：直接读取压缩包进行数据分析

数据在 `tar.gz` 中无需解压也可被读取，下面是一个演示片段，直接从打包好的历史压缩文件中吸取某一币种的多日数据拼接为 Pandas DataFrame。

In [ ]:
import io
import re

def load_historical_data(data_type, exchange, market_type, symbol, target_dates):
    """
    从 backup_dir 中的压缩包里直接加载指定条件的 Parquet 数据
    """
    all_dfs = []
    for date_str in target_dates:
        matched_tars = list(backup_dir.glob(f"crypto_data_{date_str}*.tar.gz"))
        if not matched_tars:
            print(f"⚠️ 找不到日期 {date_str} 的备份压缩包")
            continue
            
        latest_tar = max(matched_tars, key=os.path.getmtime)
        print(f"📂 正在从 {latest_tar.name} 加载数据...")
        
        target_prefix = f"{data_type}/market_type={market_type}/exchange={exchange}/symbol={symbol}/date={date_str}"
        
        day_dfs = []
        with tarfile.open(latest_tar, "r:gz") as tar:
            for member in tar.getmembers():
                if member.isfile() and member.name.startswith(target_prefix) and member.name.endswith('.parquet'):
                    filestream = tar.extractfile(member)
                    if filestream:
                        df = pd.read_parquet(io.BytesIO(filestream.read()))
                        df['date'] = date_str
                        hour_match = re.search(r'(\d+)\.parquet', member.name)
                        if hour_match:
                            df['hour'] = hour_match.group(1)
                            
                        day_dfs.append(df)
        
        if day_dfs:
            day_df = pd.concat(day_dfs, ignore_index=True)
            all_dfs.append(day_df)
            print(f"   ✅ 成功加载 {date_str} 的 {len(day_dfs)} 个片段，共 {len(day_df)} 行")

    if not all_dfs:
        return pd.DataFrame()
        
    final_df = pd.concat(all_dfs, ignore_index=True)
    final_df.sort_values('local_ts', inplace=True) if 'local_ts' in final_df.columns else final_df.sort_values('exchange_ts', inplace=True)
    final_df.reset_index(drop=True, inplace=True)
    
    return final_df

# df = load_historical_data('orderbooks', 'binance', 'spot', 'BTC_USDT', ['2026-02-25'])